# 17. LangGraph

<br>

## 17-01. LangGraph의 문법

<br>

### `TypedDict`

<br>

#### `dict`와 `TypedDict`의 차이점
- **타입 검사**
  - `dict` : 런타임에 타입 검사를 하지 않음
  - `TypedDict` : 정적 타입 검사를 제공. 코드 작성 시 IDE나 타입 체커가 오류를 미리 잡아냄
- **키와 값의 타입**
  - `dict` : 키와 값의 타입을 이랍ㄴ적으로 지정
  - `TypedDict` : 각 키에 대한 구체적인 타입을 지정
- **유연성**
  - `dict` : 런타임에 키를 추가하거나 제거
  - `TypedDict` : 정의된 구조를 따라야하며, 추가적인 키는 타입 오류를 발생

<br>

#### `TypedDict` 가 `dict` 대신 사용되는 이유
- **타입 안정성**
  - `TypedDict`는 더 엄격한 타입거사를 제공하며, 잠재적인 버그를 미리 방지할 수 있음
- **코드 가독성**
  - `TypedDict`를 사용하면 딕셔너리의 구조를 명확하게 정의할 수 있어 코드의 가독성이 향상됨
- **IDE 지원**
  - `TypedDict`를 사용하면 IDE에서 자동 완성 및 타입 힌트를 더 정확하게 제공받을 수 있음
- **문서화**
  - `TypedDict`는 코드 자체가 문서의 역할을 하여 딕셔너리의 구조를 명확히 보여줌

<br>

#### 예시

```python
from typing import Dict, TypedDict

# 일반적인 파이썬 딕셔너리(dict) 사용
sample_dict: Dict[str, str] = {
    "name": "테디",
    "age": "30",  # 문자열로 저장 (dict 에서는 가능)
    "job": "개발자",
}


# TypedDict 사용
class Person(TypedDict):
    name: str
    age: int  # 정수형으로 명시
    job: str


typed_dict: Person = {"name": "셜리", "age": 25, "job": "디자이너"}
```

<br>

```python
# dict의 경우
sample_dict["age"] = 35  # 문자열에서 정수로 변경되어도 오류 없음
sample_dict["new_field"] = "추가 정보"  # 새로운 필드 추가 가능

# TypedDict의 경우
typed_dict["age"] = 35  # 정수형으로 올바르게 사용
typed_dict["age"] = "35"  # 타입 체커가 오류를 감지함
typed_dict["new_field"] = (
    "추가 정보"  # 타입 체커가 정의되지 않은 키라고 오류를 발생시킴
)
```

<br>

### `Annotated`
- 타입 힌트에 메타데이터를 추가할 수 있게 해줌

<br>

#### `Annotated` 주요 기능
1. **추가 정보 제공** : 타입 힌트에 메타데이터를 추가하여 더 상세한 정보를 제공
2. **문서화** : 코드 자체에 추가 설명을 포함시켜 문서화 효과를 얻을 수 있음
3. **유효성 검사** : 특정 라이브러리(`Pydantic` 등)와 함께 사용하여 데이터 유효성 검사를 수행할 수 있음
4. **프레임워크 지원** : 일부 프레임워크 (`LangGraph` 등)dptj `Annotated`를 사용하여 동작을 정의

<br>

#### 기본 문법
- `Type` : 기본 타입
- `medadata1`, `metadata2` , ... : 추가하고자하는 메타데이터

```python
from typing import Annotated

variable: Annotated[Type, metadata1, metadata2, ...]
```

<br>

### 사용 예시
- `Pydantic`과 함께 사용

In [5]:
from typing import Annotated, List
from pydantic import Field, BaseModel, ValidationError

In [6]:
class Employee(BaseModel):
    id: Annotated[int, Field(..., description="직원 ID")]
    name: Annotated[str, Field(..., min_length=3, max_length=50, description="이름")]
    age: Annotated[int, Field(gt=18, lt=65, description="나이 (19-64세)")]
    salary: Annotated[
        float, Field(gt=0, lt=10000, description="연봉 (단위: 만원, 최대 10억)")
    ]
    skills: Annotated[
        List[str], Field(min_items=1, max_items=10, description="보유 기술 (1-10개)")
    ]

- 유효한 데이터로 인스턴스 생성

In [7]:
try:
    valid_employee = Employee(
        id=1, name="테디노트", age=30, salary=1000, skills=["Python", "LangChain"]
    )
    print("유효한 직원 데이터:", valid_employee)
except ValidationError as e:
    print("유효성 검사 오류:", e)

유효한 직원 데이터: id=1 name='테디노트' age=30 salary=1000.0 skills=['Python', 'LangChain']


- 유효하지 않은 데이터로 인스턴스 생성 시도

In [8]:
try:
    invalid_employee = Employee(
        name="테디",  # 이름이 너무 짧음
        age=17,  # 나이가 범위를 벗어남
        salary=20000,  # 급여가 범위를 벗어남
        skills="Python",  # 리스트가 아님
    )
except ValidationError as e:
    print("유효성 검사 오류:")
    for error in e.errors():
        print(f"- {error['loc'][0]}: {error['msg']}")


유효성 검사 오류:
- id: Field required
- name: String should have at least 3 characters
- age: Input should be greater than 18
- salary: Input should be less than 10000
- skills: Input should be a valid list


<br>

#### LangGrpah에서의 사용(`add_messages`)
- `add_message`는 LangGraph에서 메시지를 리스트에 추가하는 함수

In [9]:
from typing import Annotated, TypedDict
from langgraph.graph import add_messages

In [10]:
class MyData(TypedDict):
    messages: Annotated[list, add_messages]

<br>

### `add_messages`
- `messages` 키는 `add_messages` 리듀서 함수로 주석이 달려 있으며, 이는 LangGraph에게 기존 목록에 새 메시지를 추가하도록 지시
- 주석이 없는 상태 키는 각 업데이트에 의해 덮여쓰여져 가장 최근의 값이 저장
- `add_messages`함수는 2개의 인자 (`left`, `right`)를 받으며, 좌, 우 메시지를 병합하는 방식으로 동작

<br>

#### 주요 기능
- 두 개의 메시지 리스트를 병합
- 기본적으로 `append-only` 상태를 유지
- 동일한 ID를 가진 메시지가 있을 경우, 새 메시지로 기존 메시지를 대체

<br>

#### 동작 방식
- `right`의 메시지 중 `left`에 동일한 ID를 가진 메시지가 있으면, `right`의 메시지로 대체
- 그 외의 경우 `right`의 메시지가 `left`에 추가

<br>

#### 매개변수
- `left` (Messages) : 기본 메시지 리스트
- `right` (Messages) : 병합할 메시지 리스트 또는 단일 메시지

<br>

#### 반환값
- `Messages` : `right`의 메시지들이 `left`에 병합된 새로운 메시지 리스트

In [11]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import add_messages

In [12]:
msgs1 = [HumanMessage(content="안녕하세요?", id="1")]
msgs2 = [AIMessage(content="반갑습니다~", id="2")]

result1 = add_messages(msgs1, msgs2)
print(result1)

[HumanMessage(content='안녕하세요?', additional_kwargs={}, response_metadata={}, id='1'), AIMessage(content='반갑습니다~', additional_kwargs={}, response_metadata={}, id='2', tool_calls=[], invalid_tool_calls=[])]


<br>

- 동일한 ID를 가진 메시지의 경우

In [13]:
msgs1 = [HumanMessage(content="안녕하세요?", id="1")]
msgs2 = [HumanMessage(content="반갑습니다~", id="1")]

result2 = add_messages(msgs1, msgs2)
print(result2)

[HumanMessage(content='반갑습니다~', additional_kwargs={}, response_metadata={}, id='1')]


<br>

<hr>

<br>

## 17-02. LangGraph를 활용한 챗봇 구축

In [14]:
from dotenv import load_dotenv

load_dotenv()

True

<br>

### 1. 상태 (State) 정의

In [15]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

- 메시지 정의 : `list type` 이며 `add_messages` 함수를 사용하여 메시지를 추가

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

<br>

### 2. 노드(Node) 정의

- 노드는 작업의 단위를 나타내며, 일반적으로 정규 Python 함수

In [17]:
from langchain_openai import ChatOpenAI

In [18]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

- 챗봇 함수 정의: 현재 State를 입력으로 받아 `"messages"`라는 키 아래에 업데이트된 `messages` 목록을 포함하는 TypedDict을 반환

In [19]:
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

<br>

### 3. 그래프(Graph) 정의, 노드 추가
- 그래프 생성

In [20]:
graph_builder = StateGraph(State)

- 노드 이름, 함수 혹은 callable 객체를 인자로 받아 노드를 추가

In [21]:
graph_builder.add_node("chatbot", chatbot)